# Comparison of Human Annotation vs. LLM Codings

This notebook compares **Human Annotations** against **LLM Coding datasets** both **by Experiment (`Exp 1`, `Exp 2`, `Exp 3` — pooled across instruction conditions)** and **by Experiment & Condition (`exp1 curv`, `exp1 noInstr`, `exp2 curv`, `exp2 noInstr`, `exp3 curv`, `exp3 noInstr`)**:

1. **`human`** (Ground Truth / Reference): `Curvature Open Answers.xlsx`
2. **`detailed`**: `Curvature_Open_Answers_LLM_coded_detailed.xlsx` (LLM with detailed prompt)
3. **`minimal`**: `Curvature_Open_Answers_LLM_coded_minimal.xlsx` (LLM with minimal prompt)
4. *(Commented out)* **`old`**: `Curvature_Open_Answers_AI_old.xlsx` (`LLM Old`)

### Experiment Features:
- **Exp 1 (`exp1 curv`, `exp1 noInstr`)**: `smoothness`, `vertices`, `angle`, `intuition`, `globalness`, `size`
- **Exp 2 & Exp 3 (`exp2 curv`, `exp2 noInstr`, `exp3 curv`, `exp3 noInstr`)**: `smoothness`, `vertices`, `range`, `intuition`, `globalness`, `animal-like`, `size`

### Analysis Sections:
1. **Setup & Universal Multi-Sheet Data Loader**
2. **Overall & Experiment-Level Performance vs. Human (`Exp 1`, `Exp 2`, `Exp 3` & By Condition)**
3. **Feature-Level Agreement & Cohen's $\kappa$ by Experiment (`Exp 1`, `Exp 2`, `Exp 3`) and by Condition**
4. **Feature Prevalence Comparison (Human vs. LLMs)**
5. **Confusion Matrices (Human vs. Detailed, Minimal)**
6. **Participant-Level Discrepancy Breakdown & Export**

In [30]:
# ============================================================
# 1. Setup & Data Loader Across All Experiments (Exp 1, 2, 3)
# ============================================================
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score, confusion_matrix, f1_score

warnings.filterwarnings('ignore')

# Resolve base directory whether notebook is run from repo root or Open_Answers/
BASE_DIR = "Open_Answers" if os.path.exists("Open_Answers/Curvature Open Answers.xlsx") else "."

FILES = {
    "human": os.path.join(BASE_DIR, "Curvature Open Answers.xlsx"),
    "detailed": os.path.join(BASE_DIR, "Curvature_Open_Answers_LLM_coded_detailed.xlsx"),
    "minimal": os.path.join(BASE_DIR, "Curvature_Open_Answers_LLM_coded_minimal.xlsx"),
    # "old": os.path.join(BASE_DIR, "Curvature_Open_Answers_AI_old.xlsx"),
}

LLM_MODELS = [
    "detailed",
    "minimal",
    # "old",
]

MODEL_LABELS = {
    "human": "Human",
    "detailed": "LLM Detailed",
    "minimal": "LLM Minimal",
    # "old": "LLM Old",
}

EXPERIMENT_FEATURES = {
    "exp1 curv": ["smoothness", "vertices", "angle", "intuition", "globalness", "size"],
    "exp1 noInstr": ["smoothness", "vertices", "angle", "intuition", "globalness", "size"],
    "exp2 curv": ["smoothness", "vertices", "range", "intuition", "globalness", "animal-like", "size"],
    "exp2 noInstr": ["smoothness", "vertices", "range", "intuition", "globalness", "animal-like", "size"],
    "exp3 curv": ["smoothness", "vertices", "range", "intuition", "globalness", "animal-like", "size"],
    "exp3 noInstr": ["smoothness", "vertices", "range", "intuition", "globalness", "animal-like", "size"],
}

# Group condition sheets into top-level Experiments (Exp 1, Exp 2, Exp 3)
POOLED_EXPERIMENT_GROUPS = {
    "Exp 1": {
        "sheets": ["exp1 curv", "exp1 noInstr"],
        "features": ["smoothness", "vertices", "angle", "intuition", "globalness", "size"],
    },
    "Exp 2": {
        "sheets": ["exp2 curv", "exp2 noInstr"],
        "features": ["smoothness", "vertices", "range", "intuition", "globalness", "animal-like", "size"],
    },
    "Exp 3": {
        "sheets": ["exp3 curv", "exp3 noInstr"],
        "features": ["smoothness", "vertices", "range", "intuition", "globalness", "animal-like", "size"],
    },
}

COLUMN_ALIASES = {
    "number_of_vertices": "vertices",
    "mention_nv": "vertices",
    "mention_sm": "smoothness",
    "mention_ang": "angle",
    "animal_like": "animal-like",
}

def load_sheet(file_path, sheet_name, features):
    df = pd.read_excel(file_path, sheet_name=sheet_name, engine="openpyxl")
    df.columns = df.columns.astype(str).str.strip()
    df = df.rename(columns={k: v for k, v in COLUMN_ALIASES.items() if k in df.columns})
    
    # Standardize text column name to 'response_text'
    if "strategy_open" in df.columns:
        df["response_text"] = df["strategy_open"].fillna("").astype(str).str.strip()
    elif "open_answer" in df.columns:
        df["response_text"] = df["open_answer"].fillna("").astype(str).str.strip()
    else:
        df["response_text"] = ""
        
    df["subj_id"] = df["subj_id"].astype(str).str.strip()
    df = df[df["subj_id"].str.isdigit()].copy()
    
    for f in features:
        if f in df.columns:
            df[f] = pd.to_numeric(df[f], errors="coerce").astype("Int64")
    return df

# Detect which configured experiment sheets are present across all active files
available_sheets = {k: set(pd.ExcelFile(p).sheet_names) for k, p in FILES.items()}
EXPERIMENTS = []
for exp in EXPERIMENT_FEATURES:
    missing_in = [k for k, s_set in available_sheets.items() if exp not in s_set]
    if not missing_in:
        EXPERIMENTS.append(exp)
    else:
        print(f"Note: Sheet '{exp}' not yet present in {missing_in} — skipping until added.")

# Load all datasets across active experiment-condition sheets
data = {exp: {} for exp in EXPERIMENTS}
aligned_data = {}

for exp in EXPERIMENTS:
    feats = EXPERIMENT_FEATURES[exp]
    for key, path in FILES.items():
        data[exp][key] = load_sheet(path, exp, feats)
        
    # Merge human with active LLM models on subj_id
    h_df = data[exp]["human"][["subj_id", "response_text"] + feats].rename(
        columns={f: f"{f}_human" for f in feats}
    )
    merged = h_df.copy()
    for m in LLM_MODELS:
        m_df = data[exp][m].copy()
        extra_cols = [c for c in ["rationale", "uncertain"] if c in m_df.columns]
        rename_map = {f: f"{f}_{m}" for f in feats}
        rename_map.update({c: f"{c}_{m}" for c in extra_cols})
        m_sub = m_df[["subj_id"] + feats + extra_cols].rename(columns=rename_map)
        merged = pd.merge(merged, m_sub, on="subj_id", how="inner")
    merged["condition_sheet"] = exp
    aligned_data[exp] = merged

# Build pooled experiment datasets (Exp 1, Exp 2, Exp 3 combining curv + noInstr)
pooled_aligned_data = {}
for exp_group, info in POOLED_EXPERIMENT_GROUPS.items():
    present_sheets = [s for s in info["sheets"] if s in aligned_data]
    if present_sheets:
        pooled_aligned_data[exp_group] = pd.concat(
            [aligned_data[s] for s in present_sheets], ignore_index=True
        )

# Print loading summary
load_summary = []
for exp in EXPERIMENTS:
    row_summary = {
        "Experiment & Condition": exp,
        "Human Rows": len(data[exp]["human"]),
        "Detailed Rows": len(data[exp]["detailed"]),
        "Minimal Rows": len(data[exp]["minimal"]),
        # "Old Rows": len(data[exp]["old"]),
        "Matched Subjects": len(aligned_data[exp]),
        "Features Compared": ", ".join(EXPERIMENT_FEATURES[exp]),
    }
    load_summary.append(row_summary)
display(pd.DataFrame(load_summary))


,Experiment & Condition,Human Rows,Detailed Rows,Minimal Rows,Matched Subjects,Features Compared
0,exp1 curv,35,35,35,35,"smoothness, vertices, angle, intuition, globalness, size"
1,exp1 noInstr,36,36,36,36,"smoothness, vertices, angle, intuition, globalness, size"
2,exp2 curv,36,36,36,36,"smoothness, vertices, range, intuition, globalness, animal-like, size"
3,exp2 noInstr,36,36,36,36,"smoothness, vertices, range, intuition, globalness, animal-like, size"
4,exp3 curv,36,36,36,36,"smoothness, vertices, range, intuition, globalness, animal-like, size"
5,exp3 noInstr,36,36,36,36,"smoothness, vertices, range, intuition, globalness, animal-like, size"


## 2. Summary Performance: By Experiment (`Exp 1`, `Exp 2`, `Exp 3`), By Condition, and Overall

Computes **Raw Agreement (%)**, **Cohen's Kappa ($\kappa$)**, and **F1 Score** (treating Human as ground truth):
1. **By Experiment (`Exp 1`, `Exp 2`, `Exp 3`)** — combining `curv` + `noInstr` participants within each experiment.
2. **By Experiment & Condition (`exp1 curv`, `exp1 noInstr`, `exp2 curv`, `exp2 noInstr`, `exp3 curv`, `exp3 noInstr`)**.
3. **Overall Average Across All Experiments**.

In [31]:
def interpret_kappa(k):
    if k is None or np.isnan(k):
        return "N/A"
    if k >= 0.81:
        return "Almost Perfect"
    elif k >= 0.61:
        return "Substantial"
    elif k >= 0.41:
        return "Moderate"
    elif k >= 0.21:
        return "Fair"
    elif k >= 0.0:
        return "Slight"
    else:
        return "Poor (<0)"

def safe_kappa(v1, v2):
    if len(v1) == 0:
        return np.nan
    if (v1 == v2).all():
        return 1.0
    if v1.nunique() < 2 and v2.nunique() < 2:
        return 1.0 if (v1.iloc[0] == v2.iloc[0]) else 0.0
    return cohen_kappa_score(v1, v2)

def compute_metrics_table(dataset_dict, features_lookup, group_col_name="Experiment"):
    rows = []
    for grp_name, merged in dataset_dict.items():
        feats = features_lookup[grp_name]
        for f in feats:
            for m in LLM_MODELS:
                sub = merged[[f"{f}_human", f"{f}_{m}"]].dropna()
                vh = sub[f"{f}_human"].astype(int)
                vm = sub[f"{f}_{m}"].astype(int)
                
                agr = (vh == vm).mean() * 100
                kappa = safe_kappa(vh, vm)
                f1 = f1_score(vh, vm, zero_division=0)
                
                both_1 = int(((vh == 1) & (vm == 1)).sum())
                both_0 = int(((vh == 0) & (vm == 0)).sum())
                human_only = int(((vh == 1) & (vm == 0)).sum())  # False Negative
                llm_only = int(((vh == 0) & (vm == 1)).sum())    # False Positive
                
                rows.append({
                    group_col_name: grp_name,
                    "Feature": f,
                    "LLM_Model": m,
                    "Model_Label": MODEL_LABELS[m],
                    "N": len(sub),
                    "Agreement_Pct": agr,
                    "Kappa": kappa,
                    "Interpretation": interpret_kappa(kappa),
                    "F1": f1,
                    "Both_1": both_1,
                    "Both_0": both_0,
                    "Human_Only (1->0)": human_only,
                    "LLM_Only (0->1)": llm_only,
                })
    return pd.DataFrame(rows)

# A. Metrics by Pooled Experiment (Exp 1, Exp 2, Exp 3)
pooled_features_lookup = {k: v["features"] for k, v in POOLED_EXPERIMENT_GROUPS.items() if k in pooled_aligned_data}
pooled_metrics_df = compute_metrics_table(pooled_aligned_data, pooled_features_lookup, group_col_name="Experiment")

by_exp_summary = (
    pooled_metrics_df.groupby(["Experiment", "Model_Label"], sort=False)
    .agg(
        N_Participants=("N", "max"),
        Mean_Agreement_Pct=("Agreement_Pct", "mean"),
        Mean_Kappa=("Kappa", "mean"),
        Mean_F1=("F1", "mean"),
        Total_False_Neg=("Human_Only (1->0)", "sum"),
        Total_False_Pos=("LLM_Only (0->1)", "sum"),
    )
    .reset_index()
)
by_exp_summary["Mean_Agreement_Pct"] = by_exp_summary["Mean_Agreement_Pct"].map(lambda x: f"{x:.1f}%")
by_exp_summary["Mean_Kappa"] = by_exp_summary["Mean_Kappa"].round(3)
by_exp_summary["Mean_F1"] = by_exp_summary["Mean_F1"].round(3)

print("=== 1. Performance by Experiment (Exp 1, Exp 2, Exp 3 — Pooled Across Conditions) ===")
display(by_exp_summary)

# B. Metrics by Experiment & Condition (exp1 curv, exp1 noInstr, etc.)
metrics_df = compute_metrics_table(aligned_data, EXPERIMENT_FEATURES, group_col_name="Experiment")

exp_summary = (
    metrics_df.groupby(["Experiment", "Model_Label"], sort=False)
    .agg(
        N_Participants=("N", "max"),
        Mean_Agreement_Pct=("Agreement_Pct", "mean"),
        Mean_Kappa=("Kappa", "mean"),
        Mean_F1=("F1", "mean"),
        Total_False_Neg=("Human_Only (1->0)", "sum"),
        Total_False_Pos=("LLM_Only (0->1)", "sum"),
    )
    .reset_index()
)
exp_summary["Mean_Agreement_Pct"] = exp_summary["Mean_Agreement_Pct"].map(lambda x: f"{x:.1f}%")
exp_summary["Mean_Kappa"] = exp_summary["Mean_Kappa"].round(3)
exp_summary["Mean_F1"] = exp_summary["Mean_F1"].round(3)

print("\n=== 2. Performance by Experiment & Condition (Human vs. LLMs) ===")
display(exp_summary)

# C. Overall average across all experiments
overall_summary = (
    pooled_metrics_df.groupby("Model_Label", sort=False)
    .agg(
        Overall_Agreement_Pct=("Agreement_Pct", "mean"),
        Overall_Kappa=("Kappa", "mean"),
        Overall_F1=("F1", "mean"),
        Total_Discrepancies=("Human_Only (1->0)", lambda s: int(s.sum() + pooled_metrics_df.loc[s.index, "LLM_Only (0->1)"].sum())),
        Human_Only_FN=("Human_Only (1->0)", "sum"),
        LLM_Only_FP=("LLM_Only (0->1)", "sum"),
    )
    .reset_index()
)
overall_summary["Overall_Agreement_Pct"] = overall_summary["Overall_Agreement_Pct"].map(lambda x: f"{x:.1f}%")
overall_summary["Overall_Kappa"] = overall_summary["Overall_Kappa"].round(3)
overall_summary["Overall_F1"] = overall_summary["Overall_F1"].round(3)

print("\n=== 3. Overall Performance Averaged Across All Experiments ===")
display(overall_summary)


=== 1. Performance by Experiment (Exp 1, Exp 2, Exp 3 — Pooled Across Conditions) ===


,Experiment,Model_Label,N_Participants,Mean_Agreement_Pct,Mean_Kappa,Mean_F1,Total_False_Neg,Total_False_Pos
0,Exp 1,LLM Detailed,71,93.9%,0.833,0.886,11,15
1,Exp 1,LLM Minimal,71,81.7%,0.562,0.731,36,42
2,Exp 2,LLM Detailed,72,91.9%,0.771,0.835,15,26
3,Exp 2,LLM Minimal,72,85.7%,0.600,0.697,24,48
4,Exp 3,LLM Detailed,72,95.0%,0.846,0.883,12,13
5,Exp 3,LLM Minimal,72,86.5%,0.661,0.767,18,50



=== 2. Performance by Experiment & Condition (Human vs. LLMs) ===


,Experiment,Model_Label,N_Participants,Mean_Agreement_Pct,Mean_Kappa,Mean_F1,Total_False_Neg,Total_False_Pos
0,exp1 curv,LLM Detailed,35,94.3%,0.832,0.891,6,6
1,exp1 curv,LLM Minimal,35,80.0%,0.483,0.686,19,23
2,exp1 noInstr,LLM Detailed,36,93.5%,0.825,0.872,5,9
3,exp1 noInstr,LLM Minimal,36,83.3%,0.596,0.739,17,19
4,exp2 curv,LLM Detailed,36,93.7%,0.792,0.840,6,10
5,exp2 curv,LLM Minimal,36,87.7%,0.625,0.708,9,22
6,exp2 noInstr,LLM Detailed,36,90.1%,0.741,0.823,9,16
7,exp2 noInstr,LLM Minimal,36,83.7%,0.552,0.662,15,26
8,exp3 curv,LLM Detailed,36,96.8%,0.896,0.917,5,3
9,exp3 curv,LLM Minimal,36,88.5%,0.580,0.657,13,16



=== 3. Overall Performance Averaged Across All Experiments ===


,Model_Label,Overall_Agreement_Pct,Overall_Kappa,Overall_F1,Total_Discrepancies,Human_Only_FN,LLM_Only_FP
0,LLM Detailed,93.6%,0.816,0.867,92,38,54
1,LLM Minimal,84.8%,0.610,0.732,218,78,140


## 3A. Feature-Level Agreement & Cohen's Kappa ($\kappa$) by Experiment (`Exp 1`, `Exp 2`, `Exp 3`)

Side-by-side comparison of **Agreement (%)** and **Cohen's Kappa ($\kappa$)** against **Human Annotation** for each feature at the **Experiment level (`Exp 1`, `Exp 2`, `Exp 3` — combining `curv` and `noInstr`)**.

In [ ]:
for exp_group, feats in pooled_features_lookup.items():
    sub = pooled_metrics_df[pooled_metrics_df["Experiment"] == exp_group]
    n_subj = int(sub["N"].max()) if not sub.empty else 0
    rows = []
    for f in feats:
        f_sub = sub[sub["Feature"] == f].set_index("LLM_Model")
        row_item = {
            "Feature": f,
            "Detailed Agr (%)": f"{f_sub.loc['detailed', 'Agreement_Pct']:.1f}%",
            "Detailed Kappa": round(f_sub.loc["detailed", "Kappa"], 3),
            "Minimal Agr (%)": f"{f_sub.loc['minimal', 'Agreement_Pct']:.1f}%",
            "Minimal Kappa": round(f_sub.loc["minimal", "Kappa"], 3),
            # "Old Agr (%)": f"{f_sub.loc['old', 'Agreement_Pct']:.1f}%",
            # "Old Kappa": round(f_sub.loc["old", "Kappa"], 3),
        }
        rows.append(row_item)
    print(f"=== [{exp_group} (N={n_subj})] Feature-Level Agreement & Cohen's Kappa vs. Human ===")
    display(pd.DataFrame(rows).set_index("Feature"))

# Plot Cohen's Kappa vs Human by Experiment (Exp 1, Exp 2, Exp 3)
n_exp_groups = len(pooled_features_lookup)
fig, axes = plt.subplots(1, n_exp_groups, figsize=(5.2 * n_exp_groups, 4.6), sharey=True)
if n_exp_groups == 1:
    axes = [axes]
colors = {
    "detailed": "#4C72B0",
    "minimal": "#DD8452",
    # "old": "#55A868",
}

for idx, (exp_group, feats) in enumerate(pooled_features_lookup.items()):
    ax = axes[idx]
    sub = pooled_metrics_df[pooled_metrics_df["Experiment"] == exp_group]
    x = np.arange(len(feats))
    n_models = len(LLM_MODELS)
    width = 0.35 if n_models == 2 else 0.25
    
    for m_idx, m in enumerate(LLM_MODELS):
        offset = (m_idx - (n_models - 1) / 2) * width
        m_vals = [sub[(sub["Feature"] == f) & (sub["LLM_Model"] == m)]["Kappa"].values[0] for f in feats]
        ax.bar(x + offset, m_vals, width, label=MODEL_LABELS[m], color=colors[m], alpha=0.9)
        
    ax.axhline(0.61, color="gray", linestyle="--", linewidth=1, alpha=0.7, label="Substantial (0.61)" if idx == 0 else "")
    ax.set_title(f"{exp_group} (Pooled): Cohen's Kappa vs. Human", fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(feats, rotation=25, ha="right", fontsize=10)
    ax.set_ylim(-0.1, 1.08)
    if idx == 0:
        ax.set_ylabel("Cohen's Kappa ($\\kappa$)")
        ax.legend(frameon=True, loc="lower right")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


## 3B. Feature-Level Agreement & Cohen's Kappa ($\kappa$) by Experiment & Condition

Side-by-side comparison of **Agreement (%)** and **Cohen's Kappa ($\kappa$)** against **Human Annotation** separated by each individual condition sheet (`exp1 curv`, `exp1 noInstr`, `exp2 curv`, `exp2 noInstr`, `exp3 curv`, `exp3 noInstr`).

In [ ]:
for exp in EXPERIMENTS:
    sub = metrics_df[metrics_df["Experiment"] == exp]
    feats = EXPERIMENT_FEATURES[exp]
    
    rows = []
    for f in feats:
        f_sub = sub[sub["Feature"] == f].set_index("LLM_Model")
        row_item = {
            "Feature": f,
            "Detailed Agr (%)": f"{f_sub.loc['detailed', 'Agreement_Pct']:.1f}%",
            "Detailed Kappa": round(f_sub.loc["detailed", "Kappa"], 3),
            "Minimal Agr (%)": f"{f_sub.loc['minimal', 'Agreement_Pct']:.1f}%",
            "Minimal Kappa": round(f_sub.loc["minimal", "Kappa"], 3),
            # "Old Agr (%)": f"{f_sub.loc['old', 'Agreement_Pct']:.1f}%",
            # "Old Kappa": round(f_sub.loc["old", "Kappa"], 3),
        }
        rows.append(row_item)
    print(f"=== [{exp}] Agreement & Cohen's Kappa vs. Human ===")
    display(pd.DataFrame(rows).set_index("Feature"))

# Plot Cohen's Kappa vs Human across all condition sheets
ncols = 2
nrows = int(np.ceil(len(EXPERIMENTS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows), sharey=True)
axes = np.array(axes).flatten()

for idx, exp in enumerate(EXPERIMENTS):
    ax = axes[idx]
    sub = metrics_df[metrics_df["Experiment"] == exp]
    feats = EXPERIMENT_FEATURES[exp]
    x = np.arange(len(feats))
    n_models = len(LLM_MODELS)
    width = 0.35 if n_models == 2 else 0.25
    
    for m_idx, m in enumerate(LLM_MODELS):
        offset = (m_idx - (n_models - 1) / 2) * width
        m_vals = [sub[(sub["Feature"] == f) & (sub["LLM_Model"] == m)]["Kappa"].values[0] for f in feats]
        ax.bar(x + offset, m_vals, width, label=MODEL_LABELS[m], color=colors[m], alpha=0.9)
        
    ax.axhline(0.61, color="gray", linestyle="--", linewidth=1, alpha=0.7, label="Substantial (0.61)" if idx == 0 else "")
    ax.set_title(f"{exp}: Cohen's Kappa vs. Human", fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(feats, rotation=20, ha="right", fontsize=10)
    ax.set_ylim(-0.1, 1.08)
    ax.set_ylabel("Cohen's Kappa ($\\kappa$)")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    if idx == 0:
        ax.legend(frameon=True, loc="lower right")

for j in range(len(EXPERIMENTS), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


## 4. Feature Prevalence Comparison (Human vs. LLMs)

Compares how frequently each feature is coded as `1` (present) by **Human** vs. **`detailed`** and **`minimal`** in each experiment condition.

In [ ]:
ncols = 2
nrows = int(np.ceil(len(EXPERIMENTS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4.5 * nrows), sharey=True)
axes = np.array(axes).flatten()
all_sources = [
    "human",
    "detailed",
    "minimal",
    # "old",
]
bar_colors = {
    "human": "#222222",
    "detailed": "#4C72B0",
    "minimal": "#DD8452",
    # "old": "#55A868",
}

for idx, exp in enumerate(EXPERIMENTS):
    merged = aligned_data[exp]
    feats = EXPERIMENT_FEATURES[exp]
    
    prev_rows = []
    for f in feats:
        row_dict = {"Feature": f}
        for src in all_sources:
            col = merged[f"{f}_{src}"].dropna().astype(int)
            cnt = int(col.sum())
            pct = col.mean() * 100
            row_dict[f"{MODEL_LABELS[src]}"] = f"{cnt} ({pct:.1f}%)"
            row_dict[f"_{src}_pct"] = pct
        prev_rows.append(row_dict)
        
    prev_df = pd.DataFrame(prev_rows)
    print(f"=== [{exp}] Feature Prevalence: Count (% Coded 1) ===")
    display(prev_df[["Feature"] + [MODEL_LABELS[s] for s in all_sources]].set_index("Feature"))
    
    # Plot bars
    ax = axes[idx]
    x = np.arange(len(feats))
    n_src = len(all_sources)
    width = 0.25 if n_src == 3 else 0.2
    for s_idx, src in enumerate(all_sources):
        offset = (s_idx - (n_src - 1) / 2) * width
        ax.bar(
            x + offset,
            prev_df[f"_{src}_pct"],
            width,
            label=MODEL_LABELS[src],
            color=bar_colors[src],
            alpha=0.9,
        )
    ax.set_title(f"{exp}: Feature Prevalence (%)", fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(feats, rotation=20, ha="right", fontsize=10)
    ax.set_ylim(0, 105)
    ax.set_ylabel("% Coded as Present (1)")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    if idx == 0:
        ax.legend(frameon=True)

for j in range(len(EXPERIMENTS), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


## 5. Confusion Matrices vs. Human (Select Experiment or Condition)

Change `SELECTED_EXPERIMENT` below to any pooled experiment (`'Exp 1'`, `'Exp 2'`, `'Exp 3'`) OR any specific condition sheet (`'exp1 curv'`, `'exp1 noInstr'`, `'exp2 curv'`, `'exp2 noInstr'`, `'exp3 curv'`, `'exp3 noInstr'`) to inspect the $2 \times 2$ confusion matrices of **Human (Rows)** vs. each active **LLM Model (Columns)**.

In [ ]:
# Choose pooled experiment ('Exp 1', 'Exp 2', 'Exp 3') or condition sheet ('exp1 curv', 'exp1 noInstr', etc.)
SELECTED_EXPERIMENT = "Exp 1"

if SELECTED_EXPERIMENT in pooled_aligned_data:
    merged_sel = pooled_aligned_data[SELECTED_EXPERIMENT]
    feats_sel = pooled_features_lookup[SELECTED_EXPERIMENT]
else:
    merged_sel = aligned_data[SELECTED_EXPERIMENT]
    feats_sel = EXPERIMENT_FEATURES[SELECTED_EXPERIMENT]

n_models = len(LLM_MODELS)
fig, axes = plt.subplots(len(feats_sel), n_models, figsize=(3.8 * n_models, 2.8 * len(feats_sel)))
if n_models == 1:
    axes = axes[:, np.newaxis]

for r_idx, f in enumerate(feats_sel):
    for c_idx, m in enumerate(LLM_MODELS):
        ax = axes[r_idx, c_idx]
        sub = merged_sel[[f"{f}_human", f"{f}_{m}"]].dropna()
        vh = sub[f"{f}_human"].astype(int)
        vm = sub[f"{f}_{m}"].astype(int)
        
        cm = confusion_matrix(vh, vm, labels=[0, 1])
        ax.imshow(cm, cmap="Blues", interpolation="nearest")
        
        if r_idx == 0:
            ax.set_title(f"Human vs. {MODEL_LABELS[m]}", fontsize=11, fontweight="bold")
        if c_idx == 0:
            ax.set_ylabel(f"{f}\nHuman Code", fontsize=10, fontweight="bold")
        else:
            ax.set_ylabel("Human Code", fontsize=9)
            
        ax.set_xlabel(f"{MODEL_LABELS[m]} Code", fontsize=9)
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(["0", "1"])
        ax.set_yticklabels(["0", "1"])
        
        thresh = cm.max() / 2.0
        for i in range(2):
            for j in range(2):
                val = cm[i, j]
                ax.text(
                    j, i, f"{val}",
                    ha="center", va="center",
                    color="white" if val > thresh else "black",
                    fontsize=11, fontweight="bold"
                )

plt.suptitle(f"Confusion Matrices [{SELECTED_EXPERIMENT}]: Human (Rows) vs. LLMs (Cols)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## 6. Participant-Level Discrepancy Breakdown & Multi-Sheet Export

Shows the 100% agreement rate both **by Experiment (`Exp 1`, `Exp 2`, `Exp 3`)** and **by Experiment & Condition**, lists participant-level discrepancies, and saves the full report to `Open_Answers/Curvature_Open_Answers_Human_vs_LLMs_Comparison.xlsx`.

In [ ]:
# 1. 100% agreement overview by Pooled Experiment (Exp 1, Exp 2, Exp 3)
pooled_agreement_overview = []
for exp_group, merged in pooled_aligned_data.items():
    feats = pooled_features_lookup[exp_group]
    total_subj = len(merged)
    perfect_counts = {}
    for m in LLM_MODELS:
        mismatches = pd.Series(0, index=merged.index)
        for f in feats:
            v_h = merged[f"{f}_human"]
            v_m = merged[f"{f}_{m}"]
            both_valid = v_h.notna() & v_m.notna()
            mismatches += (both_valid & (v_h != v_m)).astype(int)
        perfect_counts[m] = int((mismatches == 0).sum())
    pooled_agreement_overview.append({
        "Experiment": exp_group,
        "Total Participants": total_subj,
        "100% Match (Detailed vs Human)": f"{perfect_counts['detailed']}/{total_subj} ({perfect_counts['detailed']/total_subj:.1%})",
        "100% Match (Minimal vs Human)": f"{perfect_counts['minimal']}/{total_subj} ({perfect_counts['minimal']/total_subj:.1%})",
        # "100% Match (Old vs Human)": f"{perfect_counts['old']}/{total_subj} ({perfect_counts['old']/total_subj:.1%})",
    })

print("=== 1. Participants with 100% Agreement by Experiment (Exp 1, Exp 2, Exp 3) ===")
display(pd.DataFrame(pooled_agreement_overview))

# 2. 100% agreement overview & discrepancy sheets by Experiment & Condition
discrepancy_sheets = {}
agreement_overview = []

for exp in EXPERIMENTS:
    merged = aligned_data[exp]
    feats = EXPERIMENT_FEATURES[exp]
    total_subj = len(merged)
    
    perfect_counts = {}
    for m in LLM_MODELS:
        mismatches = pd.Series(0, index=merged.index)
        for f in feats:
            v_h = merged[f"{f}_human"]
            v_m = merged[f"{f}_{m}"]
            both_valid = v_h.notna() & v_m.notna()
            mismatches += (both_valid & (v_h != v_m)).astype(int)
        perfect_counts[m] = int((mismatches == 0).sum())
        
    agreement_overview.append({
        "Experiment & Condition": exp,
        "Total Participants": total_subj,
        "100% Match (Detailed vs Human)": f"{perfect_counts['detailed']}/{total_subj} ({perfect_counts['detailed']/total_subj:.1%})",
        "100% Match (Minimal vs Human)": f"{perfect_counts['minimal']}/{total_subj} ({perfect_counts['minimal']/total_subj:.1%})",
        # "100% Match (Old vs Human)": f"{perfect_counts['old']}/{total_subj} ({perfect_counts['old']/total_subj:.1%})",
    })
    
    rows = []
    for _, row in merged.iterrows():
        diff_items = []
        for f in feats:
            vh = row[f"{f}_human"]
            vd = row[f"{f}_detailed"]
            vm = row[f"{f}_minimal"]
            # vo = row[f"{f}_old"]
            if pd.isna(vh):
                continue
            if any(pd.notna(v) and v != vh for v in [vd, vm]):
                diff_items.append(f"{f}: H={vh} (Det={vd}, Min={vm})")
                # diff_items.append(f"{f}: H={vh} (Det={vd}, Min={vm}, Old={vo})")
        if diff_items:
            rows.append({
                "subj_id": row["subj_id"],
                "discrepancies (H=Human, Det=Detailed, Min=Minimal)": " | ".join(diff_items),
                "response_text": row["response_text"],
                "rationale_detailed": row.get("rationale_detailed", ""),
                "rationale_minimal": row.get("rationale_minimal", ""),
            })
    discrepancy_sheets[exp] = pd.DataFrame(rows)

print("\n=== 2. Participants with 100% Agreement by Experiment & Condition ===")
display(pd.DataFrame(agreement_overview))

# Preview discrepancies
preview_sheet = SELECTED_EXPERIMENT if SELECTED_EXPERIMENT in discrepancy_sheets else EXPERIMENTS[0]
print(f"\n=== Discrepancies Preview [{preview_sheet}] ({len(discrepancy_sheets[preview_sheet])} participants with >=1 difference) ===")
pd.set_option("display.max_colwidth", None)
display(discrepancy_sheets[preview_sheet].head(15))

# Export comprehensive comparison & discrepancies workbook to Open_Answers/
export_path = os.path.join(BASE_DIR, "Curvature_Open_Answers_Human_vs_LLMs_Comparison.xlsx")
with pd.ExcelWriter(export_path, engine="openpyxl") as writer:
    overall_summary.to_excel(writer, sheet_name="Overall_Summary", index=False)
    by_exp_summary.to_excel(writer, sheet_name="By_Experiment_Summary", index=False)
    pooled_metrics_df.to_excel(writer, sheet_name="By_Experiment_Features", index=False)
    exp_summary.to_excel(writer, sheet_name="By_Condition_Summary", index=False)
    metrics_df.to_excel(writer, sheet_name="By_Condition_Features", index=False)
    for exp, d_df in discrepancy_sheets.items():
        d_df.to_excel(writer, sheet_name=f"Diff_{exp}", index=False)

print(f"\nSaved full comparison workbook to: {export_path}")
